In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

import torch
import torch.nn as nn

from torch.utils.data import DataLoader

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [ ]:
from pathlib import Path
import sys

from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

# Project path
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/project-1/mango-disease-prediction"
)

# Processed data path
PROCESSED_DIR = (
    PROJECT_ROOT / "data" / "processed"
)

print("Project exists:", PROJECT_ROOT.exists())
print("Processed directory exists:", PROCESSED_DIR.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project exists: True
Processed directory exists: True


In [ ]:
for file in [
    "paired_metadata.csv",
    "train.csv",
    "validation.csv",
    "test.csv"
]:
    path = PROCESSED_DIR / file

    print(
        file,
        "→",
        path.exists()
    )

paired_metadata.csv → True
train.csv → True
validation.csv → True
test.csv → True


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


Load the processed datasets

In [ ]:
import pandas as pd

train_df = pd.read_csv(
    PROCESSED_DIR / "train.csv"
)

val_df = pd.read_csv(
    PROCESSED_DIR / "validation.csv"
)

test_df = pd.read_csv(
    PROCESSED_DIR / "test.csv"
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (2800, 14)
Validation: (600, 14)
Test: (600, 14)


Add our project to Python

In [ ]:
PROJECT_ROOT_STR = str(PROJECT_ROOT)

if PROJECT_ROOT_STR not in sys.path:
    sys.path.insert(0, PROJECT_ROOT_STR)

print("Project added to Python path.")

Project added to Python path.


Import environmental preprocessing

In [ ]:
from src.preprocessing.environmental_preprocessing import (
    fit_preprocessors,
    transform_environmental_data
)

print("Environmental preprocessing imported.")

Environmental preprocessing imported.


In [ ]:
fit_preprocessors(train_df)

print("Preprocessor fitted using training data only.")

Preprocessor fitted using training data only.


In [ ]:
train_env = transform_environmental_data(
    train_df
)

val_env = transform_environmental_data(
    val_df
)

test_env = transform_environmental_data(
    test_df
)

print("Train environment:", train_env.shape)
print("Validation environment:", val_env.shape)
print("Test environment:", test_env.shape)

Train environment: (2800, 7)
Validation environment: (600, 7)
Test environment: (600, 7)


In [ ]:
import torch

train_env = torch.tensor(
    train_env,
    dtype=torch.float32
)

val_env = torch.tensor(
    val_env,
    dtype=torch.float32
)

test_env = torch.tensor(
    test_env,
    dtype=torch.float32
)

In [ ]:
train_labels = torch.tensor(
    train_df["label"].values,
    dtype=torch.long
)

val_labels = torch.tensor(
    val_df["label"].values,
    dtype=torch.long
)

test_labels = torch.tensor(
    test_df["label"].values,
    dtype=torch.long
)

In [ ]:
print("Train environment:", train_env.shape)
print("Train labels:", train_labels.shape)

Train environment: torch.Size([2800, 7])
Train labels: torch.Size([2800])


In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

Using device: cuda
GPU: Tesla T4


In [ ]:
train_env = train_env.to(device)
val_env = val_env.to(device)
test_env = test_env.to(device)

train_labels = train_labels.to(device)
val_labels = val_labels.to(device)
test_labels = test_labels.to(device)

In [ ]:
import torch.nn as nn


class EnvironmentalBaseline(nn.Module):

    def __init__(
        self,
        input_dim,
        num_classes
    ):
        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                input_dim,
                64
            ),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(
                64,
                32
            ),

            nn.ReLU(),

            nn.Linear(
                32,
                num_classes
            )
        )

    def forward(self, x):

        return self.network(x)

In [ ]:
NUM_CLASSES = 8

environment_input_dim = train_env.shape[1]

environment_model = EnvironmentalBaseline(
    input_dim=environment_input_dim,
    num_classes=NUM_CLASSES
).to(device)

print(environment_model)

EnvironmentalBaseline(
  (network): Sequential(
    (0): Linear(in_features=7, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Linear(in_features=32, out_features=8, bias=True)
  )
)


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    environment_model.parameters(),
    lr=0.001
)

In [ ]:
EPOCHS = 30

for epoch in range(EPOCHS):

    environment_model.train()

    optimizer.zero_grad()

    outputs = environment_model(
        train_env
    )

    loss = criterion(
        outputs,
        train_labels
    )

    loss.backward()

    optimizer.step()


    # Validation
    environment_model.eval()

    with torch.no_grad():

        val_outputs = environment_model(
            val_env
        )

        val_predictions = torch.argmax(
            val_outputs,
            dim=1
        )

        val_accuracy = (
            val_predictions == val_labels
        ).float().mean().item()


    if (epoch + 1) % 5 == 0:

        print(
            f"Epoch [{epoch+1}/{EPOCHS}] "
            f"Loss: {loss.item():.4f} "
            f"Val Accuracy: {val_accuracy:.4f}"
        )

Epoch [5/30] Loss: 2.0828 Val Accuracy: 0.1317
Epoch [10/30] Loss: 2.0788 Val Accuracy: 0.1300
Epoch [15/30] Loss: 2.0771 Val Accuracy: 0.1383
Epoch [20/30] Loss: 2.0761 Val Accuracy: 0.1317
Epoch [25/30] Loss: 2.0737 Val Accuracy: 0.1350
Epoch [30/30] Loss: 2.0725 Val Accuracy: 0.1317


In [ ]:
environment_model.eval()

with torch.no_grad():

    test_outputs = environment_model(
        test_env
    )

    test_predictions = torch.argmax(
        test_outputs,
        dim=1
    )

    test_accuracy = (
        test_predictions == test_labels
    ).float().mean().item()

print(
    f"Environmental-only Test Accuracy: "
    f"{test_accuracy:.4f}"
)

Environmental-only Test Accuracy: 0.1650


Step 1 — Import image preprocessing and Dataset

In [ ]:
%%writefile /content/drive/MyDrive/project-1/mango-disease-prediction/src/data/dataset.py

import torch

from torch.utils.data import Dataset

from PIL import Image


class MangoDiseaseDataset(Dataset):

    def __init__(
        self,
        dataframe,
        image_transform=None,
        environmental_data=None
    ):

        self.dataframe = dataframe.reset_index(
            drop=True
        )

        self.image_transform = image_transform

        self.environmental_data = environmental_data


    def __len__(self):

        return len(self.dataframe)


    def __getitem__(self, index):

        # ==========================================
        # IMAGE
        # ==========================================

        image_path = self.dataframe.iloc[
            index
        ]["image_path"]

        image = Image.open(
            image_path
        ).convert("RGB")


        # Apply image transformation
        if self.image_transform is not None:

            image = self.image_transform(
                image
            )


        # ==========================================
        # LABEL
        # ==========================================

        label = int(
            self.dataframe.iloc[
                index
            ]["label"]
        )

        label = torch.tensor(
            label,
            dtype=torch.long
        )


        # ==========================================
        # ENVIRONMENT
        # ==========================================

        if self.environmental_data is not None:

            environment = self.environmental_data[
                index
            ]

            if not isinstance(
                environment,
                torch.Tensor
            ):

                environment = torch.tensor(
                    environment,
                    dtype=torch.float32
                )

            else:

                environment = environment.float()

        else:

            # Empty tensor for image-only models
            environment = torch.empty(
                0,
                dtype=torch.float32
            )


        # ==========================================
        # RETURN SAMPLE
        # ==========================================

        return {
            "image": image,
            "environment": environment,
            "label": label
        }

Overwriting /content/drive/MyDrive/project-1/mango-disease-prediction/src/data/dataset.py


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/project-1/mango-disease-prediction"
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("Project exists:", PROJECT_ROOT.exists())
print("Processed exists:", PROCESSED_DIR.exists())

Project exists: True
Processed exists: True


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/project-1/mango-disease-prediction"
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Project exists:", PROJECT_ROOT.exists())
print("src exists:", (PROJECT_ROOT / "src").exists())

Project root: /content/drive/MyDrive/project-1/mango-disease-prediction
Project exists: True
src exists: True


In [ ]:
import src.data.dataset as dataset_module

print("Dataset file:")
print(dataset_module.__file__)

print("\nMangoDiseaseDataset exists:")
print(hasattr(dataset_module, "MangoDiseaseDataset"))

Dataset file:
/content/drive/MyDrive/project-1/mango-disease-prediction/src/data/dataset.py

MangoDiseaseDataset exists:
True


In [ ]:
from src.data.dataset import MangoDiseaseDataset

print("✅ MangoDiseaseDataset imported successfully!")

✅ MangoDiseaseDataset imported successfully!


In [ ]:
sys.path.insert(0, str(PROJECT_ROOT))

Step 3 — Check image preprocessing

In [ ]:
import src.preprocessing.image_preprocessing as image_module

print("Image preprocessing file:")
print(image_module.__file__)

print("\ntrain_transform exists:")
print(hasattr(image_module, "train_transform"))

print("val_test_transform exists:")
print(hasattr(image_module, "val_test_transform"))

Image preprocessing file:
/content/drive/MyDrive/project-1/mango-disease-prediction/src/preprocessing/image_preprocessing.py

train_transform exists:
True
val_test_transform exists:
True


In [ ]:
from src.preprocessing.image_preprocessing import (
    train_transform,
    val_test_transform
)

print("✅ Image preprocessing imported successfully!")

✅ Image preprocessing imported successfully!


Step 4 — Load the train/validation/test files

In [ ]:
import pandas as pd

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

train_df = pd.read_csv(
    PROCESSED_DIR / "train.csv"
)

val_df = pd.read_csv(
    PROCESSED_DIR / "validation.csv"
)

test_df = pd.read_csv(
    PROCESSED_DIR / "test.csv"
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 2800
Validation: 600
Test: 600


step 5 — Create image datasets

In [ ]:
train_image_dataset = MangoDiseaseDataset(
    dataframe=train_df,
    image_transform=train_transform,
    environmental_data=None
)

val_image_dataset = MangoDiseaseDataset(
    dataframe=val_df,
    image_transform=val_test_transform,
    environmental_data=None
)

test_image_dataset = MangoDiseaseDataset(
    dataframe=test_df,
    image_transform=val_test_transform,
    environmental_data=None
)

print("Train dataset:", len(train_image_dataset))
print("Validation dataset:", len(val_image_dataset))
print("Test dataset:", len(test_image_dataset))

Train dataset: 2800
Validation dataset: 600
Test dataset: 600


In [ ]:
sample = train_image_dataset[0]

print("Keys:", sample.keys())
print("Image shape:", sample["image"].shape)
print("Environment shape:", sample["environment"].shape)
print("Label:", sample["label"])

Keys: dict_keys(['image', 'environment', 'label'])
Image shape: torch.Size([3, 224, 224])
Environment shape: torch.Size([0])
Label: tensor(4)


1. Create the DataLoaders

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_image_loader = DataLoader(
    train_image_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_image_loader = DataLoader(
    val_image_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_image_loader = DataLoader(
    test_image_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("✅ DataLoaders created")

✅ DataLoaders created


In [ ]:
batch = next(iter(train_image_loader))

print("Image:", batch["image"].shape)
print("Environment:", batch["environment"].shape)
print("Label:", batch["label"].shape)

Image: torch.Size([32, 3, 224, 224])
Environment: torch.Size([32, 0])
Label: torch.Size([32])


1. Import ViT

In [ ]:
import torch
import torch.nn as nn

from torchvision.models import (
    vit_b_16,
    ViT_B_16_Weights
)

vit_model = vit_b_16(
    weights=ViT_B_16_Weights.DEFAULT
)

# Change ImageNet's 1000 classes
# to our 8 mango disease classes
vit_model.heads.head = nn.Linear(
    vit_model.heads.head.in_features,
    8
)

vit_model = vit_model.to(device)

print("✅ ViT created")
print(vit_model.heads)

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:06<00:00, 52.6MB/s]


✅ ViT created
Sequential(
  (head): Linear(in_features=768, out_features=8, bias=True)
)


. Create a simpler DataLoader

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 16

train_image_loader = DataLoader(
    train_image_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

val_image_loader = DataLoader(
    val_image_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

test_image_loader = DataLoader(
    test_image_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

print("✅ New DataLoaders created")

✅ New DataLoaders created


In [ ]:
import time

start = time.time()

sample = train_image_dataset[0]

elapsed = time.time() - start

print("Image shape:", sample["image"].shape)
print("Label:", sample["label"])
print("Time:", round(elapsed, 2), "seconds")

Image shape: torch.Size([3, 224, 224])
Label: tensor(4)
Time: 0.01 seconds


In [ ]:
start = time.time()

batch = next(iter(train_image_loader))

elapsed = time.time() - start

print("Image shape:", batch["image"].shape)
print("Label shape:", batch["label"].shape)
print("Batch loading time:", round(elapsed, 2), "seconds")

Image shape: torch.Size([16, 3, 224, 224])
Label shape: torch.Size([16])
Batch loading time: 2.04 seconds


In [ ]:
sample = train_image_dataset[0]
print(sample["image"].shape)

torch.Size([3, 224, 224])


In [ ]:
batch = next(iter(train_image_loader))
print(batch["image"].shape)

torch.Size([16, 3, 224, 224])


In [ ]:
!nvidia-smi

Mon Aug 31 10:15:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P0             29W /   70W |    5307MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Validate the ViT

In [ ]:
vit_model.eval()

correct = 0
total = 0

all_val_predictions = []
all_val_labels = []

with torch.no_grad():

    for batch in val_image_loader:

        images = batch["image"].to(
            device,
            non_blocking=True
        )

        labels = batch["label"].to(
            device,
            non_blocking=True
        )

        outputs = vit_model(images)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        total += labels.size(0)

        correct += (
            predictions == labels
        ).sum().item()

        all_val_predictions.extend(
            predictions.cpu().numpy()
        )

        all_val_labels.extend(
            labels.cpu().numpy()
        )


val_accuracy = correct / total

print(
    f"ViT Validation Accuracy: "
    f"{val_accuracy:.4f}"
)

In [ ]:
import torch
import torch.nn as nn

from torchvision.models import (
    vit_b_16,
    ViT_B_16_Weights
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [ ]:
vit_model = vit_b_16(
    weights=ViT_B_16_Weights.DEFAULT
)

# Change ImageNet's 1000-class classifier
# to our 8 mango disease classes
vit_model.heads.head = nn.Linear(
    vit_model.heads.head.in_features,
    8
)

vit_model = vit_model.to(device)

print(vit_model.heads)

Sequential(
  (head): Linear(in_features=768, out_features=8, bias=True)
)


In [ ]:
for param in vit_model.parameters():
    param.requires_grad = False

for param in vit_model.heads.head.parameters():
    param.requires_grad = True

print("✅ ViT backbone frozen")
print("✅ Classification head is trainable")

✅ ViT backbone frozen
✅ Classification head is trainable


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    vit_model.heads.head.parameters(),
    lr=0.001,
    weight_decay=0.01
)

print("✅ Optimizer ready")

✅ Optimizer ready


In [ ]:
import time

batch = next(iter(train_image_loader))

images = batch["image"].to(
    device
)

labels = batch["label"].to(
    device
)

start = time.time()

vit_model.train()

outputs = vit_model(images)

loss = criterion(
    outputs,
    labels
)

loss.backward()

optimizer.step()

elapsed = time.time() - start

print("✅ One ViT batch completed")
print("Input:", images.shape)
print("Output:", outputs.shape)
print("Loss:", loss.item())
print("Time:", round(elapsed, 2), "seconds")

✅ One ViT batch completed
Input: torch.Size([16, 3, 224, 224])
Output: torch.Size([16, 8])
Loss: 2.006192445755005
Time: 0.04 seconds


Step 1 — Train for 3 epochs

In [ ]:
print("ViT exists:", "vit_model" in globals())
print("Train loader exists:", "train_image_loader" in globals())
print("Device:", device)
print("CUDA:", torch.cuda.is_available())

ViT exists: False
Train loader exists: True
Device: cuda
CUDA: True


In [ ]:
import time
import torch
import torch.nn as nn
from torchvision.models import (
    vit_b_16,
    ViT_B_16_Weights
)

# Re-initialize device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Re-initialize vit_model
vit_model = vit_b_16(
    weights=ViT_B_16_Weights.DEFAULT
)
NUM_CLASSES = 8 # Assuming NUM_CLASSES is 8 based on previous context
vit_model.heads.head = nn.Linear(
    vit_model.heads.head.in_features,
    NUM_CLASSES
)

vit_model = vit_model.to(device)

# Freeze backbone layers
for param in vit_model.parameters():
    param.requires_grad = False

# Make classification head trainable
for param in vit_model.heads.head.parameters():
    param.requires_grad = True

# Re-initialize criterion and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    vit_model.heads.head.parameters(),
    lr=0.001,
    weight_decay=0.01
)

EPOCHS = 3

for epoch in range(EPOCHS):

    vit_model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    start_time = time.time()

    for batch_idx, batch in enumerate(train_image_loader):

        images = batch["image"].to(
            device,
            non_blocking=True
        )

        labels = batch["label"].to(
            device,
            non_blocking=True
        )

        # Clear gradients
        optimizer.zero_grad()

        # Forward
        outputs = vit_model(images)

        # Loss
        loss = criterion(
            outputs,
            labels
        )

        # Backward
        loss.backward()

        # Update classifier
        optimizer.step()

        # Statistics
        running_loss += loss.item()

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        total += labels.size(0)

        correct += (
            predictions == labels
        ).sum().item()

        # Progress
        if (batch_idx + 1) % 10 == 0:

            print(
                f"Epoch {epoch + 1}/{EPOCHS} | "
                f"Batch {batch_idx + 1}/{len(train_image_loader)} | "
                f"Loss: {loss.item():.4f}"
            )

    train_accuracy = correct / total

    elapsed = time.time() - start_time

    print(
        f"\nEpoch {epoch + 1} finished"
    )

    print(
        f"Training Loss: "
        f"{running_loss / len(train_image_loader):.4f}"
    )

    print(
        f"Training Accuracy: "
        f"{train_accuracy:.4f}"
    )

    print(
        f"Time: {elapsed / 60:.2f} minutes\n"
    )

Epoch 1/3 | Batch 10/175 | Loss: 1.3015
Epoch 1/3 | Batch 20/175 | Loss: 1.0786
Epoch 1/3 | Batch 30/175 | Loss: 0.4681
Epoch 1/3 | Batch 40/175 | Loss: 0.5218
Epoch 1/3 | Batch 50/175 | Loss: 0.2722
Epoch 1/3 | Batch 60/175 | Loss: 0.3039
Epoch 1/3 | Batch 70/175 | Loss: 0.3617
Epoch 1/3 | Batch 80/175 | Loss: 0.2088
Epoch 1/3 | Batch 90/175 | Loss: 0.1756
Epoch 1/3 | Batch 100/175 | Loss: 0.1306
Epoch 1/3 | Batch 110/175 | Loss: 0.0869
Epoch 1/3 | Batch 120/175 | Loss: 0.1455
Epoch 1/3 | Batch 130/175 | Loss: 0.0670
Epoch 1/3 | Batch 140/175 | Loss: 0.0878
Epoch 1/3 | Batch 150/175 | Loss: 0.1821
Epoch 1/3 | Batch 160/175 | Loss: 0.0849
Epoch 1/3 | Batch 170/175 | Loss: 0.0792

Epoch 1 finished
Training Loss: 0.3831
Training Accuracy: 0.9182
Time: 19.79 minutes

Epoch 2/3 | Batch 10/175 | Loss: 0.0556
Epoch 2/3 | Batch 20/175 | Loss: 0.0383
Epoch 2/3 | Batch 30/175 | Loss: 0.0879
Epoch 2/3 | Batch 40/175 | Loss: 0.0608
Epoch 2/3 | Batch 50/175 | Loss: 0.0560
Epoch 2/3 | Batch 60/175 

Validate the ViT

In [ ]:
vit_model.eval()

correct = 0
total = 0

all_val_predictions = []
all_val_labels = []

with torch.no_grad():

    for batch in val_image_loader:

        images = batch["image"].to(
            device,
            non_blocking=True
        )

        labels = batch["label"].to(
            device,
            non_blocking=True
        )

        outputs = vit_model(images)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        total += labels.size(0)

        correct += (
            predictions == labels
        ).sum().item()

        all_val_predictions.extend(
            predictions.cpu().numpy()
        )

        all_val_labels.extend(
            labels.cpu().numpy()
        )


val_accuracy = correct / total

print(
    f"ViT Validation Accuracy: "
    f"{val_accuracy:.4f}"
)

ViT Validation Accuracy: 0.9950


Test the ViT

In [ ]:
vit_model.eval()

correct = 0
total = 0

all_test_predictions = []
all_test_labels = []

with torch.no_grad():

    for batch in test_image_loader:

        images = batch["image"].to(
            device,
            non_blocking=True
        )

        labels = batch["label"].to(
            device,
            non_blocking=True
        )

        outputs = vit_model(images)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        total += labels.size(0)

        correct += (
            predictions == labels
        ).sum().item()

        all_test_predictions.extend(
            predictions.cpu().numpy()
        )

        all_test_labels.extend(
            labels.cpu().numpy()
        )


vit_test_accuracy = correct / total

print(
    f"ViT Test Accuracy: "
    f"{vit_test_accuracy:.4f}"
)

ViT Test Accuracy: 0.9967


Classification report

In [ ]:
from sklearn.metrics import classification_report

CLASS_NAMES = [
    "Anthracnose",
    "Bacterial Canker",
    "Cutting Weevil",
    "Die Back",
    "Gall Midge",
    "Healthy",
    "Powdery Mildew",
    "Sooty Mould"
]

print(
    classification_report(
        all_test_labels,
        all_test_predictions,
        target_names=CLASS_NAMES
    )
)

                  precision    recall  f1-score   support

     Anthracnose       1.00      0.99      0.99        75
Bacterial Canker       1.00      1.00      1.00        75
  Cutting Weevil       1.00      1.00      1.00        75
        Die Back       1.00      1.00      1.00        75
      Gall Midge       1.00      0.99      0.99        75
         Healthy       0.97      1.00      0.99        75
  Powdery Mildew       1.00      1.00      1.00        75
     Sooty Mould       1.00      1.00      1.00        75

        accuracy                           1.00       600
       macro avg       1.00      1.00      1.00       600
    weighted avg       1.00      1.00      1.00       600

